In [1]:
import socket
import cv2
import numpy as np
import time
import threading

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
leds = base.leds

### Season Colors in HEX

In [2]:
spring_hex =   ["FBCEB1",
                "FFDAB3",
                "F88379",
                "FFF1C1",
                "FFFF99",
                "C4E1C1",
                "B3E3E2",
                "FFE28A",
                "FADADD",
                "FFB7A5",
                "D6C9FF",
                "F7E7CE",
                "FFBF00",
                "007FFF",
                "FF6F61",
                "4CBB17",
                "FF6347",
                "FFD700",
                "00A86B",
                "FF9966",
                "FFA500",
                "FF5C7A",
                "0F52BA",
                "40E0D0",
                "FF5E5B",
                "FFF700",
                "FF9900",
                "FF8C69",
                "7FFF00",
                "00CED1",
                "FF69B4",
                "FFFF00",
                "00FFFF",
                "FF6347",
                "DAA520",
                "A7FC00"]

summer_hex = ["B4CFEC",
              "FFD1DC",
              "B5EAD7",
              "C8A2C8",
              "C8A2C8",
              "FFF1C1",
              "E6E6FA",
              "B0E0E6",
              "98FF98",
              "F7CAC9",
              "87CEEB",
              "CCCCFF",
              "D4A5A5",
              "BCB88A",
              "6A5ACD",
              "E0B0FF",
              "C2B280",
              "483C32",
              "6B7B8C",
              "B39EB5",
              "8A9A5B",
              "B38481",
              "708090",
              "5E7D7E",
              "3B5998",
              "3E8E8E",
              "8E4585",
              "71797E",
              "C76B8A",
              "4A7C6F",
              "7366BD",
              "4F84C4",
              "6B4570",
              "5B8A8A",
              "5A6D7A",
              "9B3A62"]

autumn_hex =   ["D4AF37",
                "D8A48F",
                "A3AA8B",
                "C87C5A",
                "808000",
                "D2B1A3",
                "9B6A6C",
                "F5DEB3",
                "E2725B",
                "967969",
                "8A9A5B",
                "B08D57",
                "DAA520",
                "FF7518",
                "B87333",
                "E34234",
                "F4C430",
                "B7410E",
                "A0522D",
                "738678",
                "FFC512",
                "CD7F32",
                "6B8E23",
                "A0522D",
                "4A1C17",
                "556B2F",
                "4D5D53",
                "8A3324",
                "8B2500",
                "645452",
                "773F1A",
                "9B4722",
                "800000",
                "228B22",
                "5C3317",
                "996515"]

winter_hex =   ["007FFF",
                "990066",
                "800020",
                "36454F",
                "A5F2F3",
                "26619C",
                "FF00FF",
                "0F52BA",
                "8A2BE2",
                "C21E56",
                "708090",
                "CCCCFF",
                "0055FF",
                "FF69B4",
                "E0115F",
                "000000",
                "E5F9F9",
                "A020F0",
                "D9D9D6",
                "00FFFF",
                "D200D2",
                "3F00FF",
                "C0C0C0",
                "1C1C1C",
                "000080",
                "9B111E",
                "005F5F",
                "004B49",
                "4A0E0E",
                "580F41",
                "082567",
                "2B2B2B",
                "8B008B",
                "722F37",
                "2F4F4F",
                "004953"]

### Color Analysis Functions

In [3]:
def bgr_to_lab(color):
    bgr = np.array(color, dtype=np.uint8).reshape(1,1,3)  # shape (1,1,3)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    return tuple(lab[0,0])

In [44]:
OFFSET = 135

def chroma(a,b):
    return np.sqrt((a-OFFSET)**2 + (b-OFFSET)**2)

def determine_season_lab(colors):
    skin, hair, eye = colors

    sL,sA,sB = skin
    hL,hA,hB = hair
    eL,eA,eB = eye

    # hue (warm vs cool)
    warm_score = 0.25*(sA-OFFSET) + (sB-OFFSET)
    print(f"warmth: {warm_score}")
#     hue = "warm" if warm_score >= 0 else "cool"

    # chroma (bright vs muted)
    avg_chroma = (0.6*chroma(sA,sB) +
                  0.3*chroma(hA,hB) +
                  0.1*chroma(eA,eB))

    chroma_type = "bright" if avg_chroma >= 10 else "muted"
    print(f"chroma avg: {avg_chroma}")

    # value contrast
    contrast = max(sL,hL,eL) - min(sL,hL,eL)

    if warm_score < 2:
        hue = "cool" if contrast > 50 else "warm"
    else: hue = "warm"
    print(f"hue: {hue}, contrast: {contrast}, chroma: {chroma_type}")
        
    # season
    if hue == "cool":
        if int(contrast) > 45:
            season = "winter"
        elif int(contrast) < 35:
            season = "summer"
        else:
            if chroma_type == "bright":
                season = "winter"
            else: # chroma = muted
                season = "summer"
    else: # hue == "warm":
        if chroma_type == "bright":
            season = "spring"
        else: # chroma == muted
            season = "autumn"
    
    return season

In [5]:
 def all_onboard_leds_off():
    for a in leds:
        a.off()

### LED Display Functions

In [6]:
%%microblaze base.PMODA

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
int write_gpio_a(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

void clear_gpios_a(){
    for(int i = 0; i < 8; ++i){
        write_gpio_a(i,0);
    }
}

In [7]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
int write_gpio_b(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

void clear_gpios_b(){
    for(int i = 0; i < 8; ++i){
        write_gpio_b(i,0);
    }
}

In [8]:
def hex_to_pwm_duty(hex_code, gamma=2.2):
    """
    Convert a 6-character hex color code to perceptually corrected PWM duty cycles (0-100)
    
    Args:
        hex_code (str): 6-character hex string, e.g., 'FF7A3C'
        gamma (float): gamma correction factor (default 2.2)
        
    Returns:
        tuple: (R_duty, G_duty, B_duty) each 0-100
    """
    if len(hex_code) != 6:
        raise ValueError("Hex code must be 6 characters, e.g., 'A1B2C3'.")

    try:
        # Convert hex to 0-255
        r = int(hex_code[0:2], 16)
        g = int(hex_code[2:4], 16)
        b = int(hex_code[4:6], 16)

        # Normalize to 0-1
        r_norm = r / 255
        g_norm = g / 255
        b_norm = b / 255

        # Apply gamma correction
        r_gamma = r_norm ** gamma
        g_gamma = g_norm ** gamma
        b_gamma = b_norm ** gamma

        # Scale to 0-100 for duty cycle
        r_duty = int(r_gamma * 100)
        g_duty = int(g_gamma * 100)
        b_duty = int(b_gamma * 100)

        return (r_duty, g_duty, b_duty)

    except ValueError:
        raise ValueError("Invalid hex characters. Must be 0-9 or A-F.")

In [9]:
def pwm_thread_pmoda():
    leds = [0,1]
    while not stop_event.is_set():
        for led in leds:
            for color in ["R","G","B"]:
                pin = pins[led][color]
                on_time = duty[led][color] / 100 / freq
                off_time = (1 - duty[led][color] / 100) / freq
                if on_time > 0:
                    write_gpio_a(pin, 1)
                    time.sleep(on_time)
                write_gpio_a(pin, 0)
                time.sleep(off_time)

def pwm_thread_pmodb():
    leds = [2,3]
    while not stop_event.is_set():
        for led in leds:
            for color in ["R","G","B"]:
                pin = pins[led][color]
                on_time = duty[led][color] / 100 / freq
                off_time = (1 - duty[led][color] / 100) / freq
                if on_time > 0:
                    write_gpio_b(pin, 1)
                    time.sleep(on_time)
                write_gpio_b(pin, 0)
                time.sleep(off_time)

In [10]:
def color_cycler(season_list):
    """
    sliding window to show colors in palette
    """
    idx = [0,1,2,3]
    n_colors = len(season_list)

    while not stop_event.is_set():
        for led in range(4):
            r,g,b = season_list[idx[led]]
            duty[led]["R"] = r
            duty[led]["G"] = g
            duty[led]["B"] = b
            idx[led] = (idx[led] + 1) % n_colors
        time.sleep(1)  # each color stays 1 second


In [11]:
spring_rgb_pwm = []
for color in spring_hex:
    spring_rgb_pwm.append(hex_to_pwm_duty(color))

summer_rgb_pwm = []
for color in summer_hex:
    summer_rgb_pwm.append(hex_to_pwm_duty(color))

autumn_rgb_pwm = []
for color in autumn_hex:
    autumn_rgb_pwm.append(hex_to_pwm_duty(color))

winter_rgb_pwm = []
for color in winter_hex:
    winter_rgb_pwm.append(hex_to_pwm_duty(color))

### Main

In [72]:
colors = []
all_onboard_leds_off()

freq = 1300 

# mappings
pins = (
    {"R": 7, "G": 6, "B": 5},   # LED 0 (PMODA)
    {"R": 3, "G": 2, "B": 1},   # LED 1 (PMODA)
    {"R": 7, "G": 6, "B": 5},   # LED 2 (PMODB)
    {"R": 3, "G": 2, "B": 1}    # LED 3 (PMODB)
)

# duty list for each LED
duty = [
    {"R":0,"G":0,"B":0},
    {"R":0,"G":0,"B":0},
    {"R":0,"G":0,"B":0},
    {"R":0,"G":0,"B":0}
]

season_to_pwm = {
    "spring": spring_rgb_pwm,
    "summer": summer_rgb_pwm,
    "autumn": autumn_rgb_pwm,
    "winter": winter_rgb_pwm
}

# --------------------------------------------------------------------
CLIENT_IP = "0.0.0.0"
LISTENING_PORT = 9999

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.bind((CLIENT_IP, LISTENING_PORT))
sock.listen(1)

conn, addr = sock.accept()
print(f"Accepted connection from {addr}")

while True:
    data = conn.recv(1024)
    print("data comming in")
    if not data:
        print("Client disconneted")
        break
    print("Received message: ", data.decode('utf8'))
    bgr_tuple = tuple(map(int, data.split()))
    colors.append(bgr_tuple)
print("all data received") 
sock.shutdown(socket.SHUT_RDWR)
sock.close()

print(f"colors: {colors}")

# ---------------------------------------------------------------------
# skin, hair, eye in lab format
lab_colors = []
for color in colors:
    lab_colors.append(bgr_to_lab(color))
print(f"lab: {lab_colors}")

# --------------------------------------------------------
print("begin analysis")
season = determine_season_lab(lab_colors)
print(season)
selected_season_list = season_to_pwm.get(season)

led_num = -1

if season == 'spring':
    led_num = 0
elif season == 'summer':
    led_num = 1
elif season == 'autumn':
    led_num = 2
elif season == 'winter':
    led_num = 3
else: pass

for i in range(4):
    leds[led_num].on()
    time.sleep(0.5)
    leds[led_num].off()
    time.sleep(0.5)

leds[led_num].on()

stop_event = threading.Event()

threads = []

t_pmoda = threading.Thread(target=pwm_thread_pmoda)
t_pmodb = threading.Thread(target=pwm_thread_pmodb)

t_pmoda.start()
t_pmodb.start()

threads.append(t_pmoda)
threads.append(t_pmodb)

cycler_thread = threading.Thread(target=color_cycler, args=(selected_season_list,))
cycler_thread.start()

print("PWM threads running. Press Enter to stop.")
input()

stop_event.set()

cycler_thread.join()
for t in threads:
    t.join()

print("All PWM threads stopped.")

print("Done.")

Accepted connection from ('192.168.2.1', 5524)
data comming in
Received message:  96 104 141

data comming in
Received message:  31 31 32

data comming in
Received message:  83 88 126

data comming in
Client disconneted
all data received
colors: [(96, 104, 141), (31, 31, 32), (83, 88, 126)]
lab: [(121, 142, 138), (30, 128, 128), (105, 143, 137)]
begin analysis
warmth: 4.75
chroma avg: 8.363933469625376
hue: cool, contrast: 91, chroma: muted
winter
PWM threads running. Press Enter to stop.

All PWM threads stopped.
Done.


In [73]:
conn.close()
sock.close()